# Fine-tuning ESM2 with Human Spatial Constraint (HuSC)

LoRA fine-tuning of ESM2 (8M, 35M, 150M, 650M) so that the per-residue entropy of the model's amino-acid distribution ranks residues in the same order as HuSC.

**Inputs** (in `DATA_DIR`): HuSC scores (`HuSC_8_2026-10.tar.gz`), protein sequences (`UniProt_Sequences.fasta`) and ProteinGym UniProt accessions (`proteingym_uniprot_accessions.csv`) from Dryad (https://doi.org/10.5061/dryad.1vhhmgr88), and the ProteinGym substitution benchmark (`DMS_substitutions.csv` and the `DMS/` folder of assays). The train/validation and ProteinGym dev/test splits used in the paper are also on Dryad; when present they are reused instead of recomputed.

**Training data.** HuSC proteins with any ProteinGym target or close homolog (≥30% identity over ≥50% of the target) removed, split 75/25 into train/validation by 30%-identity clusters. Training uses the most constrained quarter of the training proteins (by the fraction of residues with permutation p < 0.05).

**Objective.** ListMLE between the normalized entropy of the 20-amino-acid distribution (unmasked forward pass) and HuSC, on 20 randomly sampled residues per protein per step. LoRA r = 8, α = 64, dropout 0.1 on query/key/value/output projections; AdamW, lr 1e-5, batch size 1, 2 epochs.

**Checkpoint selection.** ProteinGym assays are split by protein into a dev third and a test two-thirds. The checkpoint with the highest mean dev Spearman (evaluated every 500 proteins, patience 5) is kept. The test split is not used here; evaluation is in the analysis notebook.

**Outputs** (in `DATA_DIR`): `finetuned_esm2_{size}_husc.pth` (LoRA weights), `finetune_history_esm2_{size}_husc.csv`, `finetune_summary.csv`.

In [ ]:
#@title 1. Setup
# DATA_DIR: folder with the inputs listed above; outputs are written there too. On Colab, mount Google Drive first if the
# data are there. Needs a GPU runtime.
DATA_DIR = "HuSC_data"

!uv pip install -q biopython
# peft's LoRA dispatcher fails on the torchao version preinstalled in Colab; torchao is not used here.
!pip uninstall -y -q torchao

import os, gc, re, glob, math, random, time, subprocess
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from Bio import SeqIO
from scipy.stats import spearmanr
from transformers import EsmForMaskedLM, EsmTokenizer
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

LOCAL = "/content/husc"
HUSC_DIR = f"{LOCAL}/HuSC_8"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# HuSC scores, one file per protein, extracted to local disk (much faster than reading them from a mounted Drive).
if not os.path.isdir(HUSC_DIR):
    os.makedirs(LOCAL, exist_ok=True)
    subprocess.run(["tar", "-xzf", f"{DATA_DIR}/HuSC_8_2026-10.tar.gz", "-C", LOCAL], check=True)
    for f in os.listdir(HUSC_DIR):
        if f.startswith("._"):                       # macOS AppleDouble files from the tarball
            os.remove(os.path.join(HUSC_DIR, f))
n_husc = len([f for f in os.listdir(HUSC_DIR) if f.endswith("_husc.tsv")])
print(f"HuSC proteins: {n_husc:,}")

# MMseqs2 (release 18-8cc5c), used for the homology filters in steps 2 and 4
MMSEQS_RELEASE = "18-8cc5c"
MMSEQS = f"/content/mmseqs_{MMSEQS_RELEASE}/mmseqs/bin/mmseqs"
if not os.path.exists(MMSEQS):
    os.makedirs(f"/content/mmseqs_{MMSEQS_RELEASE}", exist_ok=True)
    subprocess.run(["wget", "-q", f"https://github.com/soedinglab/MMseqs2/releases/download/{MMSEQS_RELEASE}/mmseqs-linux-avx2.tar.gz",
                    "-O", "/content/mmseqs.tar.gz"], check=True)
    subprocess.run(["tar", "-xzf", "/content/mmseqs.tar.gz", "-C", f"/content/mmseqs_{MMSEQS_RELEASE}"], check=True)

def mmseqs_search(query_fasta, target_fasta, out_tsv):
    """All hits with E < 1e-3; coverage fractions of query and target added as qcov/tcov."""
    cols = ["query", "target", "fident", "alnlen", "qlen", "tlen", "evalue", "bits"]
    if not os.path.exists(out_tsv):
        subprocess.run([MMSEQS, "easy-search", query_fasta, target_fasta, out_tsv, f"{out_tsv}.tmp",
                        "-s", "7.5", "-e", "1e-3", "--max-seqs", "5000", "--format-output", ",".join(cols)],
                       check=True, capture_output=True)
    h = pd.read_csv(out_tsv, sep="\t", names=cols)
    h["qcov"], h["tcov"] = h.alnlen / h.qlen, h.alnlen / h.tlen
    return h

def homologous(h, min_ident=0.30, min_cov=0.50):
    """>= 30% identity over >= 50% of EITHER sequence (so a short protein matching a domain of a long one counts)."""
    return h[(h.fident >= min_ident) & ((h.qcov >= min_cov) | (h.tcov >= min_cov))]

def write_fasta(path, records):
    with open(path, "w") as f:
        f.writelines(f">{k}\n{s}\n" for k, s in records)

fasta = {r.id.split("|")[0]: str(r.seq) for r in SeqIO.parse(f"{DATA_DIR}/UniProt_Sequences.fasta", "fasta")}

In [ ]:
#@title 2. Train/validation split of HuSC proteins, excluding ProteinGym targets and their homologs
# A HuSC protein is excluded if its accession matches a ProteinGym target or it shares >= 30% identity over >= 50%
# of either sequence with one, in a search in either direction (MMseqs2 E-values depend on the database searched,
# so the two directions find different hits). The remaining proteins are clustered at 30% identity and whole
# clusters are split 75/25 into train/validation. The split is written to DATA_DIR once and reused.
SPLIT_PREFIX = f"{DATA_DIR}/husc_finetune_split"
SPLIT_FILES = {k: f"{SPLIT_PREFIX}_{k}_ids.txt" for k in ["train", "val", "excluded_proteingym"]}

if not all(os.path.exists(p) for p in SPLIT_FILES.values()):
    W = "/content/split"
    os.makedirs(W, exist_ok=True)
    husc_ids = [u for u in sorted({f.split("_")[0] for f in os.listdir(HUSC_DIR) if f.endswith("_husc.tsv")}) if u in fasta]
    pg = (pd.read_csv(f"{DATA_DIR}/DMS_substitutions.csv").drop_duplicates("DMS_id")
          .drop(columns=["uniprot_id", "real_uniprot"], errors="ignore")
          .merge(pd.read_csv(f"{DATA_DIR}/proteingym_uniprot_accessions.csv"), on="DMS_id", how="left"))
    write_fasta(f"{W}/husc_all.fasta", ((u, fasta[u]) for u in husc_ids))
    write_fasta(f"{W}/pg_targets.fasta", zip(pg.DMS_id, pg.target_seq))

    pg_accessions = set(pg.uniprot_id.dropna().astype(str)) | set(pg.real_uniprot.dropna().astype(str))
    fwd = homologous(mmseqs_search(f"{W}/pg_targets.fasta", f"{W}/husc_all.fasta", f"{W}/pg_vs_husc.tsv"))
    rev = homologous(mmseqs_search(f"{W}/husc_all.fasta", f"{W}/pg_targets.fasta", f"{W}/husc_vs_pg.tsv"))
    excluded = {u for u in husc_ids if u in pg_accessions} | set(fwd.target) | set(rev["query"])
    keep = [u for u in husc_ids if u not in excluded]

    write_fasta(f"{W}/keep.fasta", ((u, fasta[u]) for u in keep))
    subprocess.run([MMSEQS, "easy-cluster", f"{W}/keep.fasta", f"{W}/clust", f"{W}/tmp_clust",
                    "--min-seq-id", "0.3", "-c", "0.5", "--cov-mode", "0", "-s", "7.5"],
                   check=True, capture_output=True)
    cl = pd.read_csv(f"{W}/clust_cluster.tsv", sep="\t", names=["rep", "member"])
    member2rep = dict(zip(cl.member, cl.rep))
    reps = sorted(set(member2rep.values()))
    np.random.default_rng(0).shuffle(reps)
    val_reps = set(reps[: int(round(0.25 * len(reps)))])

    split = {"train": sorted(u for u in keep if member2rep.get(u, u) not in val_reps),
             "val": sorted(u for u in keep if member2rep.get(u, u) in val_reps),
             "excluded_proteingym": sorted(excluded)}
    for k, ids in split.items():
        with open(SPLIT_FILES[k], "w") as f:
            f.write("\n".join(ids))

read_ids = lambda p: [l.strip() for l in open(p) if l.strip()]
train_ids_all = read_ids(SPLIT_FILES["train"])
val_ids = read_ids(SPLIT_FILES["val"])
print(f"train {len(train_ids_all):,} | validation {len(val_ids):,} | excluded (ProteinGym) "
      f"{len(read_ids(SPLIT_FILES['excluded_proteingym'])):,}")

In [ ]:
#@title 3. Training proteins: the most constrained quarter of the training split
# frac_p05 = fraction of a protein's residues with HuSC < -1.645 (permutation p < 0.05).
STATS_PATH = f"{DATA_DIR}/husc_protein_stats_2026-10.csv"

if not os.path.exists(STATS_PATH):
    rows = []
    for fn in tqdm(sorted(f for f in os.listdir(HUSC_DIR) if f.endswith("_husc.tsv")), desc="HuSC per-protein stats"):
        v = pd.read_csv(f"{HUSC_DIR}/{fn}", sep="\t", usecols=["HuSC"]).HuSC.dropna().values
        if len(v):
            rows.append(dict(uniprot_id=fn.split("_")[0], n_res=len(v), frac_p05=float((v < -1.645).mean())))
    pd.DataFrame(rows).to_csv(STATS_PATH, index=False)

PSTATS = pd.read_csv(STATS_PATH)
pool = PSTATS[PSTATS.uniprot_id.isin(set(train_ids_all))]
cut = pool.frac_p05.quantile(0.75)
TRAIN_IDS = sorted(pool.loc[pool.frac_p05 >= cut, "uniprot_id"])
print(f"training proteins: {len(TRAIN_IDS):,} of {len(train_ids_all):,} (frac_p05 >= {cut:.3f})")

In [ ]:
#@title 4. ProteinGym dev/test split and the dev-set score used for checkpoint selection
# Assays with target length <= 1022 are split by UniProt ID, one third DEV and two thirds TEST (seed 2026). Targets are
# then grouped into homology clusters (connected components of >= 30% identity over >= 50% of either sequence), and
# every cluster that contains a DEV assay is assigned to DEV entirely, so no TEST target is homologous to a DEV target.
# Only DEV is used here (checkpoint selection); TEST is evaluated in the analysis notebook.
PG_SUMMARY = pd.read_csv(f"{DATA_DIR}/DMS_substitutions.csv").drop_duplicates("DMS_id")
DMS_CACHE = {}
for p in tqdm(sorted(glob.glob(f"{DATA_DIR}/DMS/*.csv")), desc="loading DMS assays"):
    try:
        d = pd.read_csv(p, usecols=lambda c: c in ("mutant", "DMS_score"))
    except Exception as e:
        print("skip", os.path.basename(p), e)
        continue
    if {"mutant", "DMS_score"} <= set(d.columns):
        DMS_CACHE[os.path.basename(p)[:-4]] = d

PG_SPLIT_PATH = f"{DATA_DIR}/proteingym_dev_test_split.csv"
if not os.path.exists(PG_SPLIT_PATH):
    elig = PG_SUMMARY[(PG_SUMMARY.seq_len <= 1022) & PG_SUMMARY.DMS_id.isin(DMS_CACHE.keys())].reset_index(drop=True)
    prots = sorted(elig.UniProt_ID.astype(str).unique())
    np.random.default_rng(2026).shuffle(prots)
    dev_prots = set(prots[: len(prots) // 3])
    split = dict(zip(elig.DMS_id, np.where(elig.UniProt_ID.astype(str).isin(dev_prots), "dev", "test")))

    W = "/content/pg_split"
    os.makedirs(W, exist_ok=True)
    write_fasta(f"{W}/pg.fasta", zip(elig.DMS_id, elig.target_seq))
    h = homologous(mmseqs_search(f"{W}/pg.fasta", f"{W}/pg.fasta", f"{W}/pg_vs_pg.tsv"))
    parent = {d: d for d in elig.DMS_id}                        # union-find over homologous targets
    def root(d):
        while parent[d] != d:
            parent[d] = parent[parent[d]]; d = parent[d]
        return d
    for q, t in zip(h["query"], h["target"]):
        parent[root(q)] = root(t)
    dev_roots = {root(d) for d, s in split.items() if s == "dev"}
    moved = [d for d, s in split.items() if s == "test" and root(d) in dev_roots]
    for d in moved:
        split[d] = "dev"
    print(f"moved {len(moved)} TEST assays homologous to a DEV target into DEV: {sorted(moved)}")
    pd.DataFrame(dict(DMS_id=elig.DMS_id, UniProt_ID=elig.UniProt_ID.astype(str), taxon=elig.taxon,
                      cluster=[root(d) for d in elig.DMS_id], split=elig.DMS_id.map(split))).to_csv(PG_SPLIT_PATH, index=False)

PG_SPLIT = pd.read_csv(PG_SPLIT_PATH)
PG_DEV = set(PG_SPLIT.loc[PG_SPLIT.split == "dev", "DMS_id"])
print(f"ProteinGym assays: dev {len(PG_DEV)}, test {int((PG_SPLIT.split == 'test').sum())}")

_MUT = re.compile(r"^([A-Z])(\d+)([A-Z])$")

@torch.no_grad()
def llr_matrix(seq):
    """(L, 20) log-likelihood ratio of every substitution against the wild-type residue (unmasked forward pass)."""
    enc = tokenizer(seq, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    logits = model(enc["input_ids"].to(device), attention_mask=enc["attention_mask"].to(device)).logits
    lp = torch.log_softmax(logits[0, 1:-1, AA_LO:AA_HI], dim=-1)
    wt = torch.tensor([aa_to_idx.get(c, 0) for c in seq[:lp.size(0)]], device=device)
    return (lp - lp.gather(1, wt[:, None])).float().cpu().numpy()

@torch.no_grad()
def proteingym_spearman(dms_ids, use_adapter=True):
    """Mean over assays of Spearman(DMS_score, mean LLR over the variant's substitutions)."""
    model.eval()
    rhos = []
    with (nullcontext() if use_adapter else model.disable_adapter()):
        todo = PG_SUMMARY[(PG_SUMMARY.seq_len <= 1022) & PG_SUMMARY.DMS_id.isin(dms_ids)]
        for _, r in todo.iterrows():
            dms = DMS_CACHE.get(r.DMS_id)
            if dms is None:
                continue
            seq, M = r.target_seq, llr_matrix(r.target_seq)
            scores = np.full(len(dms), np.nan)
            for i, mut in enumerate(dms.mutant.astype(str).values):
                vals = []
                for m in mut.split(":"):
                    g = _MUT.match(m)
                    if not g:
                        vals = None; break
                    wt_aa, p, mu = g.group(1), int(g.group(2)), g.group(3)
                    if not (1 <= p <= M.shape[0]) or seq[p - 1] != wt_aa or mu not in aa_to_idx:
                        vals = None; break
                    vals.append(M[p - 1, aa_to_idx[mu]])
                if vals:
                    scores[i] = np.mean(vals)
            ok = ~np.isnan(scores) & dms.DMS_score.notna().values
            if ok.sum() >= 10:
                rhos.append(spearmanr(scores[ok], dms.DMS_score.values[ok]).statistic)
    return float(np.nanmean(rhos))

In [ ]:
#@title 5. Model, data and objective
MODELS = {"8M":   "facebook/esm2_t6_8M_UR50D",
          "35M":  "facebook/esm2_t12_35M_UR50D",
          "150M": "facebook/esm2_t30_150M_UR50D",
          "650M": "facebook/esm2_t33_650M_UR50D"}
RECIPE = dict(lora_r=8, lora_alpha=64, lora_dropout=0.1, lora_targets=["query", "key", "value", "output.dense"],
              lr=1e-5, weight_decay=1e-2, epochs=2, n_residues=20, eval_every=500, patience=5, seed=0)
MAX_LEN = 1024                      # tokens, including CLS and EOS
MAX_RES = MAX_LEN - 2

tokenizer = EsmTokenizer.from_pretrained(MODELS["8M"])        # the vocabulary is identical for all ESM2 sizes
AA_LO, AA_HI = 4, 24                                          # token ids of the 20 standard amino acids
AA20 = [a for a, i in sorted(tokenizer.get_vocab().items(), key=lambda kv: kv[1]) if AA_LO <= i < AA_HI]
aa_to_idx = {a: i for i, a in enumerate(AA20)}

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def build_model(model_name):
    m = EsmForMaskedLM.from_pretrained(model_name)
    m = get_peft_model(m, LoraConfig(task_type="FEATURE_EXTRACTION", r=RECIPE["lora_r"], lora_alpha=RECIPE["lora_alpha"],
                                     target_modules=RECIPE["lora_targets"], lora_dropout=RECIPE["lora_dropout"]))
    for n, p in m.named_parameters():
        p.requires_grad = "lora" in n
    return m.to(device)

class HuSCDataset(Dataset):
    """One protein per item: tokens, 0-based positions of residues with a HuSC score, and the scores."""
    def __init__(self, ids):
        self.ids = ids
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        uid = self.ids[i]
        seq = fasta[uid][:MAX_RES]
        df = pd.read_csv(f"{HUSC_DIR}/{uid}_husc.tsv", sep="\t", usecols=["uniprot_pos", "HuSC"])
        df = df[(df.uniprot_pos >= 1) & (df.uniprot_pos <= len(seq))].dropna()
        if len(df) < 2:
            return None
        husc = torch.tensor(df.HuSC.values, dtype=torch.float32)
        enc = tokenizer(seq, return_tensors="pt", truncation=True, max_length=MAX_LEN)
        return dict(seq=seq, input_ids=enc["input_ids"][0], attention_mask=enc["attention_mask"][0],
                    pos=torch.tensor(df.uniprot_pos.values - 1, dtype=torch.long), husc=husc)

def loader(ids, generator=None):
    return DataLoader(HuSCDataset(ids), batch_size=1, shuffle=generator is not None,
                      generator=generator, collate_fn=lambda b: b[0])

def entropy_score(b):
    """Normalized Shannon entropy of the 20-amino-acid distribution at each residue (higher = more tolerant)."""
    logits = model(b["input_ids"][None].to(device), attention_mask=b["attention_mask"][None].to(device)).logits
    lp = torch.log_softmax(logits[0, 1:-1, AA_LO:AA_HI], dim=-1)
    return -(lp.exp() * lp).sum(-1) / math.log(20.0)

def listmle(pred, target):
    """Negative Plackett-Luce log-likelihood of the target order under the predicted scores."""
    p = pred[target.sort(descending=True).indices]
    return (torch.log(p.exp().flip(0).cumsum(0).flip(0) + 1e-10) - p).mean()

def batch_loss(b, gen):
    pos, tgt = b["pos"].to(device), b["husc"].to(device)
    if len(pos) > RECIPE["n_residues"]:
        sel = torch.randperm(len(pos), generator=gen, device="cpu")[:RECIPE["n_residues"]].to(device)
        pos, tgt = pos[sel], tgt[sel]
    s = entropy_score(b)
    keep = pos < s.size(0)
    if keep.sum() < 2:
        return None
    return listmle(s[pos[keep]], tgt[keep])

@torch.no_grad()
def husc_val_spearman(val_loader, use_adapter=True):
    """Mean per-protein Spearman(entropy, HuSC) on held-out HuSC proteins (logged; not used for selection)."""
    model.eval()
    rhos = []
    with (nullcontext() if use_adapter else model.disable_adapter()):
        for b in val_loader:
            if b is None:
                continue
            s = entropy_score(b)
            keep = b["pos"] < s.size(0)
            if keep.sum() < 5:
                continue
            rho = spearmanr(s[b["pos"][keep].to(device)].float().cpu().numpy(), b["husc"][keep].numpy()).statistic
            if not np.isnan(rho):
                rhos.append(rho)
    return float(np.mean(rhos))

In [ ]:
#@title 6. Training loop with ProteinGym-dev checkpoint selection
SUMMARY_PATH = f"{DATA_DIR}/finetune_summary.csv"
VAL_EVAL_IDS = sorted(np.random.default_rng(123).choice(val_ids, size=400, replace=False).tolist())

def finetune(size):
    """Train one LoRA adapter; keep the checkpoint with the best ProteinGym dev Spearman."""
    global model
    name = f"esm2_{size}_husc"
    ckpt, hist_path = f"{DATA_DIR}/finetuned_{name}.pth", f"{DATA_DIR}/finetune_history_{name}.csv"
    R = RECIPE

    set_seed(R["seed"])
    if "model" in globals():
        del model; gc.collect(); torch.cuda.empty_cache()
    model = build_model(MODELS[size])
    train_loader = loader(TRAIN_IDS, generator=torch.Generator().manual_seed(R["seed"]))
    val_loader = loader(VAL_EVAL_IDS)
    base_val = husc_val_spearman(val_loader, use_adapter=False)
    base_dev = proteingym_spearman(PG_DEV, use_adapter=False)
    print(f"\n{name}: base PG dev {base_dev:.4f} | base HuSC val {base_val:.4f}", flush=True)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=R["lr"], weight_decay=R["weight_decay"])
    sample_gen = torch.Generator().manual_seed(R["seed"])
    hist, best, best_step, bad, step, run_loss, n_loss = [], -np.inf, 0, 0, 0, 0.0, 0
    t0, stop = time.time(), False
    model.train()
    for epoch in range(R["epochs"]):
        for b in train_loader:
            if b is None:
                continue
            loss = batch_loss(b, sample_gen)
            if loss is None or torch.isnan(loss):
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); opt.zero_grad()
            step += 1; run_loss += loss.item(); n_loss += 1
            if step % R["eval_every"]:
                continue
            dev, val = proteingym_spearman(PG_DEV), husc_val_spearman(val_loader)
            model.train()
            hist.append(dict(step=step, epoch=epoch + 1, train_loss=run_loss / n_loss, pg_dev=dev, husc_val=val,
                             minutes=round((time.time() - t0) / 60, 2)))
            pd.DataFrame(hist).to_csv(hist_path, index=False)
            run_loss, n_loss = 0.0, 0
            if dev > best + 1e-4:
                best, best_step, bad = dev, step, 0
                torch.save({k: v for k, v in model.state_dict().items() if "lora" in k}, ckpt)
            else:
                bad += 1
            print(f"  step {step:5d}  loss {hist[-1]['train_loss']:.4f}  PG dev {dev:.4f}  HuSC val {val:.4f}"
                  f"{'  *' if best_step == step else ''}", flush=True)
            if bad >= R["patience"]:
                stop = True
                break
        if stop:
            break

    row = dict(run=name, size=size, base_pg_dev=base_dev, best_pg_dev=best, best_step=best_step,
               steps=step, base_husc_val=base_val,
               husc_val_at_best=next(h["husc_val"] for h in hist if h["step"] == best_step),
               minutes=round((time.time() - t0) / 60, 1), n_train=len(TRAIN_IDS))
    S = pd.read_csv(SUMMARY_PATH) if os.path.exists(SUMMARY_PATH) else pd.DataFrame()
    S = pd.concat([S[S.run != name] if len(S) else S, pd.DataFrame([row])], ignore_index=True)
    S.to_csv(SUMMARY_PATH, index=False)
    print(f"  -> {name}: PG dev {base_dev:.4f} -> {best:.4f} (step {best_step}), {row['minutes']} min", flush=True)
    return row

In [ ]:
#@title 7. Train all model sizes
done = set(pd.read_csv(SUMMARY_PATH).run) if os.path.exists(SUMMARY_PATH) else set()
for size in MODELS:
    name = f"esm2_{size}_husc"
    if name in done:
        print(f"{name}: already trained"); continue
    finetune(size)
pd.read_csv(SUMMARY_PATH)